# Analyse des commentaires
Dans ce notebook, nous allons regarder en détail les commentaires laissés par les utilisateurs.
Le travail sera divisé en deux parties : Construction du corpus et Début ? d'analyse des fréquences

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from surprise import NMF, Dataset
from surprise.reader import Reader
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist
from nltk.collocations import BigramCollocationFinder, BigramAssocMeasures, TrigramCollocationFinder, TrigramAssocMeasures
from itertools import product


from boardgames_recsys.data.filtering import filter_df
import boardgames_recsys.text.filtering as ft
from boardgames_recsys.data.matrix import *
from boardgames_recsys.models.collaborative_filtering import *
from boardgames_recsys.evaluation.ratings import *


%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
def words_freq(data, corpus) -> pd.DataFrame:
    """
    Construction d'un dataframe avec la fréquence des mots dans un corpus
    """

    lem, occurences = np.unique(data['Lemma'], return_counts=True)

    df = pd.DataFrame({'Lemma': lem, 'Freq': occurences})
    nb_comments = data["Comment line"].nunique()
    df['Freq'] = df['Freq'].apply(lambda val: val/nb_comments)

    # Garder uniquement les lemmas qui appraissent dans le corpus
    #return df[df['Lemma'].isin(corpus)]
    return df

def construction_corpus(lemmas:pd.DataFrame, taille: int) -> dict:
    """ 
    Construction d'un corpus à partir d'une BDD de commentaires
    avis.colums = 'Comment title', 'Comment body'

    Retourne df avec mots du corpus et leurs fréquences, les 'taille' plus fréquentes
    """

    # Corpus creation from lemmatized dataframe
    lemmas = lemmas[~lemmas["Lemma"].isna()]
    lemmas = lemmas[lemmas['Part of speech'].isin(['ADJ', 'NOM', "VER", "NEG"])]
    lemmas = lemmas[~lemmas["Lemma"].isin(["bref", "bof", "excelent", "bon", "autre", "seul", "tendre", "fin"
                                           "super", "superbe", "juste", "jouable", "ca", "faire", "pouvoir", "ausi"])]
    lemmas = lemmas['Lemma'].to_numpy()

    # Occurencies calculation for each lemma
    lem, occ = np.unique(lemmas, return_counts=True)
    freq_lem = pd.DataFrame({'lemma': lem, 'freq': occ})

    freq_lem = freq_lem.sort_values(by=['freq'], ascending=False)
    return freq_lem.head(taille)['lemma'].to_numpy()

In [ ]:
folder = "../database_cleaned"
avis_clean  = pd.read_csv(f"{folder}/avis_clean.csv", index_col=0)
jeux_clean  = pd.read_csv(f"{folder}/jeux_clean.csv", index_col=0)
users       = pd.read_csv(f"{folder}/users.csv", index_col=0)

min_reviews = 10 
rev_filter = filter_df(avis_clean, min_reviews)
games_means = rev_filter[["Game id", "Rating"]].groupby("Game id").mean().reset_index()

rev_filter = rev_filter.assign(index=np.arange(0, rev_filter.shape[0]))
rev_filter_center, _= center_score(rev_filter)
rev_filter_center

### Corpus 5000 mots

In [ ]:
lemmas = pd.read_csv("../generated_data/Lemmas_VER_cleaned.csv", index_col=0)
corpus = construction_corpus(lemmas, 5000) 
lemmas = lemmas[lemmas["Lemma"].isin(corpus)] # only words in corpus

# Joined lemmas
comments = lemmas.groupby("Comment line")["Lemma"].apply(" ".join).reset_index().rename(columns={"Lemma" : "Comment"})
comments

### NMF 20 latent factors

In [ ]:
model = NMF(n_factors=20, random_state=42, biased=False, reg_pu= 0.1, reg_qi= 0.1)
data = Dataset.load_from_df(rev_filter[["User id", "Game id", "Rating"]], reader=Reader(rating_scale=(0, 10)))
trainset = data.build_full_trainset()
nmf = model.fit(trainset)

# Extract matrices
U = nmf.pu  # User-feature matrix (W)
G = nmf.qi  # Item-feature matrix (H)

games_ids = np.array([trainset.to_raw_iid(i) for i in range(len(G))])
users_ids = np.array([trainset.to_raw_uid(u) for u in range(len(U))])
G = G[np.argsort(games_ids), :]
G

### 30 KMeans games clusters 

In [ ]:
sns.set_theme(rc={"figure.figsize":(6, 5)})
NB_CLUSTERS = 30
kmeans = KMeans(n_clusters=NB_CLUSTERS, random_state=42) 
kmeans.fit(G) 

games_clusters = pd.DataFrame(data={"Game id" : np.sort(games_ids), "Cluster" : kmeans.labels_})
games_clusters

In [ ]:
# Séparation de la bdd 
positifs = rev_filter_center[rev_filter_center['Rating'] >= 0]
negatifs = rev_filter_center[rev_filter_center['Rating'] < 0]

print("Nombre d'avis negatif", len(negatifs)/len(rev_filter_center))
print("Nombre d'avis positif", len(positifs)/len(rev_filter_center))

In [ ]:
lemmas_pos = positifs[["Game id", "User id", "index"]].merge(lemmas, right_on="Comment line", left_on="index")
lemmas_neg = negatifs[["Game id", "User id", "index"]].merge(lemmas, right_on="Comment line", left_on="index")
lemmas_pos = lemmas_pos.drop(["index"], axis=1)
lemmas_neg = lemmas_neg.drop(["index"], axis=1)

lemmas_all = rev_filter[["User id", "Game id", "index"]].merge(lemmas, right_on="Comment line", left_on="index")
lemmas_all = lemmas_all.drop(["index"], axis=1)
lemmas_all

In [ ]:
comments_neg = lemmas_neg.groupby(by=["Comment line", "Game id", "User id"])["Lemma"].apply(" ".join).reset_index()
comments_neg = comments_neg.assign(pos_neg = "negative")

comments_pos = lemmas_pos.groupby(by=["Comment line", "Game id", "User id"])["Lemma"].apply(" ".join).reset_index()
comments_pos = comments_pos.assign(pos_neg = "positive")

comments_all = pd.concat([comments_neg, comments_pos])
comments_all_count = comments_all[["Game id", "pos_neg", "User id"]].groupby(["Game id", "pos_neg"]).count().rename(columns={"User id":"count"}).reset_index()

In [ ]:
comments_neg[comments_neg["Lemma"].str.contains("must haver")]

In [ ]:
rev_neg_count = comments_neg["Game id"].value_counts().reset_index()
rev_pos_count = comments_pos["Game id"].value_counts().reset_index()

# Filter games so that each game has at least 10 pos and 10 neg reviews
games_preserved = rev_neg_count[rev_neg_count["Game id"].isin(rev_pos_count.loc[rev_pos_count["count"] >= 10, "Game id"])
                                & rev_neg_count["Game id"].isin(rev_neg_count.loc[rev_neg_count["count"] >= 10, "Game id"])]["Game id"].values
                                
mask = np.isin(np.sort(games_ids), games_preserved)

# Games clusters contains only games that were filtered
games_clusters = pd.DataFrame(data={"Game id":np.sort(games_ids)[mask], "Cluster":kmeans.labels_[mask]})

In [ ]:
# Barplot the distribution of pos/neg comments
def plot_pos_neg_games(selected_games, comments_all_count, title, all=False):
    sns.set_theme(rc={"figure.figsize":(15, 6)})
    filtered = comments_all_count[comments_all_count["Game id"].isin(selected_games["Game id"])]
    if not all:
        filtered = filtered.head(50)
    ax = sns.barplot(data=filtered, x="Game id", y="count", hue="pos_neg", errorbar=None)
    ax.set_title(title)
    if all:
        ax.set(xticklabels=[])

def create_df(ngram_finder, ngram_stat):
        bigram_freq = ngram_finder.score_ngrams(ngram_stat)

        bigrams_df = pd.DataFrame(data=[list(info) for info in bigram_freq])
        bigrams_df[0] = bigrams_df[0].apply(list).apply(" ".join)
        bigrams_df = bigrams_df.rename(columns={0:"Lemma", 1:"Freq"})
        return bigrams_df

def get_Ngrams(game, ngram_finder, ngram_stat):
    comments_pos = lemmas_pos[lemmas_pos["Game id"] == game].groupby("Comment line")["Lemma"].apply(list)
    comments_neg = lemmas_neg[lemmas_neg["Game id"] == game].groupby("Comment line")["Lemma"].apply(list)
    
    #if comments_pos.size > 0:
    bigram_finder_pos = ngram_finder.from_documents(comments_pos)
    freq_pos = create_df(bigram_finder_pos, ngram_stat)
    #if comments_neg.size > 0:
    bigram_finder_neg = ngram_finder.from_documents(comments_neg)
    freq_neg = create_df(bigram_finder_neg, ngram_stat)
    
    diff_freq = ft.diff_freq(freq_pos, freq_neg)

    return freq_pos, freq_neg, diff_freq

def plot_games_Ngrams_freq_diff(selected_games:np.array, nrows:int, ncols:int, figsize:tuple, ngram_finder, ngram_stat, games_means):
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    for game, (i, j) in zip(selected_games, list(product(range(0, nrows - 1, 2), range(ncols)))):
        mean = games_means[games_means["Game id"] == game]["Rating"].item()

        _, _, diff_check_games = get_Ngrams(game, ngram_finder,ngram_stat)

        sns.barplot(data=diff_check_games.head(20), y="Lemma", x="Freq differency", ax=axes[i, j])
        sns.barplot(data=diff_check_games.tail(20), y="Lemma", x="Freq differency", ax=axes[i + 1, j])

        axes[i, j].set_title(f"Game {game} head freq_diff {mean:.2f}")
        axes[i + 1, j].set_title(f"Game {game} tail freq_diff {mean:.2f}")

    plt.tight_layout()

def plot_games_Ngrams_all(selected_games:np.array, nrows:int, ncols:int, figsize:tuple, ngram_finder, ngram_stat, games_means):
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    for game, (i, j) in zip(selected_games, list(product(range(0, nrows, 2), range(ncols)))):
        
        mean = games_means[games_means["Game id"] == game]["Rating"].item()
        pos, neg, _ = get_Ngrams(game, ngram_finder,ngram_stat)

        sns.barplot(data=pos.head(20), y="Lemma", x="Freq", ax=axes[i, j])
        sns.barplot(data=neg.head(20), y="Lemma", x="Freq", ax=axes[i + 1, j])

        axes[i, j].set_title(f"Game {game} head pos {mean:.2f}")
        axes[i + 1, j].set_title(f"Game {game} head neg {mean:.2f}")

    plt.tight_layout()

### Bigrams

#### Most rated games

In [ ]:
top_games = rev_filter[["Game id", "Rating"]].groupby("Game id").count().reset_index().sort_values(by="Rating", ascending=False).head(12)

plot_pos_neg_games(top_games, comments_all_count, f"Number of pos/neg comments most rated games", all=True)

#### Most rated games frequency difference pos/neg

In [ ]:
plot_games_Ngrams_freq_diff(top_games["Game id"], 6, 4, (20, 20), BigramCollocationFinder, BigramAssocMeasures.raw_freq, games_means)

### Games per cluster

In [ ]:
CLUSTER = 11
cluster_games = games_clusters[games_clusters["Cluster"] == CLUSTER]
plot_pos_neg_games(cluster_games, comments_all_count, f"Number of pos/neg comments in cluster {CLUSTER}", all=True)

In [ ]:
plot_games_Ngrams_freq_diff(cluster_games["Game id"], 6, 4, (20, 20), BigramCollocationFinder, BigramAssocMeasures.raw_freq, games_means)

**Positifs**
- règles simples, simple efficace, jeu simple, traduction règles, fabriquer en france
- plein monstre, bon mécanique, épic quest
- rapide fun, partie court, familial, jeu apéro, cocktail

- apprendre histoire, evenement invention, acquerir connaissance, agréable manipuler
- gros jeu, jeu équilibré, très difficile

- boite metal, matériel qualité, matériel irreprochable, qualité prix, petite boite
- nouvelle version, tempete cerveau

- condition victoire

- coopératif, original

**Négatifs**
- règles simples, trop simple
- très tactique, version familial?
- gros boîte, matériel qualité (austère), casser brique (jeu de construction)
- petit jeu, gros jeu, jeu expert
- passer temps (à lire des règles)
- illustration, couleur limité
- certain déséquilibre

- extension à ajouter
- manque intéraction
- manque le profondeur

### Pos / neg bigrams on all comments

In [ ]:
all_bigrams_pos = create_df(BigramCollocationFinder.from_documents(comments_pos["Lemma"].str.split()), BigramAssocMeasures.raw_freq)
all_bigrams_neg = create_df(BigramCollocationFinder.from_documents(comments_neg["Lemma"].str.split()), BigramAssocMeasures.raw_freq)
all_bigrams_freq_diff = ft.diff_freq(all_bigrams_pos, all_bigrams_neg)

# Positive
ax = sns.lineplot(data=all_bigrams_freq_diff.head(90), x="Lemma", y="Freq differency")
ax.tick_params(axis='x', rotation=90, labelsize=8)
ax.set_title("Most frequent bigrams in positive reviews")

In [ ]:
# Negative
ax = sns.lineplot(data=all_bigrams_freq_diff.tail(90).sort_values(by="Freq differency"), x="Lemma", y="Freq differency")
ax.tick_params(axis='x', rotation=90, labelsize=8)
ax.set_title("Most frequent bigrams in negative reviews")

### User recommendation KNN

In [ ]:
# Init
matrix_ratings, mask_ratings, users_table, games_table = get_matrix_user_game(rev_filter)
cos_sim_matrix = calc_distance_matrix(matrix_ratings, mask_ratings, "cos")
top_users = rev_filter[["User id", "Rating"]].groupby("User id").count().reset_index().sort_values(by="Rating", ascending=False)["User id"].values

games_to_consider = games_clusters["Game id"].values

users_mean = rev_filter[["User id", "Rating"]].groupby("User id").mean().reset_index()
top_users[:5]

In [ ]:
# old version
def knn_similar_comments(user_id, games_to_consider, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table):

    def plot_barplots(sim_users_neg, sim_users_pos, user_neg, user_pos):

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))

        # Negatives comments
        bigrams_neg = create_df(BigramCollocationFinder.from_documents(sim_users_neg["Lemma"].str.split().tolist()),
                                BigramAssocMeasures.raw_freq)

        bigrams_neg_user = create_df( BigramCollocationFinder.from_documents(user_neg["Lemma"].str.split().tolist()),
                                BigramAssocMeasures.raw_freq)
        
        bigrams_neg = bigrams_neg[bigrams_neg["Lemma"].isin(bigrams_neg_user["Lemma"])]

        # Find intersection
        bigrams_neg = bigrams_neg[bigrams_neg["Lemma"].isin(bigrams_neg_user["Lemma"])]
        bigrams_neg_user = bigrams_neg_user[bigrams_neg_user["Lemma"].isin(bigrams_neg["Lemma"])]

        # Positive comments
        bigrams_pos = create_df(BigramCollocationFinder.from_documents(sim_users_pos["Lemma"].str.split().tolist()),
                                BigramAssocMeasures.raw_freq)

        bigrams_pos_user = create_df( BigramCollocationFinder.from_documents(user_pos["Lemma"].str.split().tolist()),
                                BigramAssocMeasures.raw_freq)
        
        # Find intersection
        bigrams_pos = bigrams_pos[bigrams_pos["Lemma"].isin(bigrams_pos_user["Lemma"])]
        bigrams_pos_user = bigrams_pos_user[bigrams_pos_user["Lemma"].isin(bigrams_pos["Lemma"])]


        sns.barplot(data=bigrams_neg.sort_values(by="Freq", ascending=False).head(40), y="Lemma", x="Freq", ax=ax1)
        sns.barplot(data=bigrams_pos.sort_values(by="Freq", ascending=False).head(40), y="Lemma", x="Freq", ax=ax2)

        sns.barplot(data=bigrams_neg_user.sort_values(by="Freq", ascending=False).head(40), y="Lemma", x="Freq", ax=ax1, color="r", alpha=0.5)
        sns.barplot(data=bigrams_pos_user.sort_values(by="Freq", ascending=False).head(40), y="Lemma", x="Freq", ax=ax2, color="r", alpha=0.5)

        ax1.set_title(f"Negative bigrams for user {user_id} (id)")
        ax2.set_title(f"Positive bigrams for user {user_id} (id)")
        ax1.tick_params(axis='y', labelsize=8)
        ax2.tick_params(axis='y', labelsize=8)

        plt.tight_layout()
        return ax1, ax2

    user_ind = users_table[users_table == user_id].index[0]
    games_to_hide = np.random.choice(games_to_consider, size=200, replace=False)

    hidden_games = np.intersect1d(games_table[games_table.isin(games_to_hide)].index, mask_ratings[user_ind, :].nonzero()[0])

    prev_ratings, prev_mask_ratings = matrix_ratings[user_ind, :], mask_ratings[user_ind, :], 
    prev_sim = cos_sim_matrix[user_ind, :]

    # hide games
    matrix_ratings[user_ind, hidden_games] = 0
    mask_ratings[user_ind, hidden_games] = 0

    recalc_cos_similarity(user_ind, matrix_ratings, cos_sim_matrix)

    sim_users =  get_KNN(cos_sim_matrix, 40, user_ind)
    print("similar users", sim_users)
    pred_ratings, mask_pred_ratings = predict_ratings_baseline(matrix_ratings, mask_ratings,
                                                                sim_users, cos_sim_matrix, user_ind)
    
    # restore
    matrix_ratings[user_ind, :], mask_ratings[user_ind, :] = prev_ratings, prev_mask_ratings
    cos_sim_matrix[user_ind, :], cos_sim_matrix[:, user_ind] = prev_sim, prev_sim

    diff = np.abs(matrix_ratings[user_ind, hidden_games] - pred_ratings[hidden_games])

    ALLOW_ERR = 2
    user_mean = users_mean.loc[users_mean["User id"] == user_id, "Rating"].item()
    pos, neg = pred_ratings[hidden_games] < user_mean, pred_ratings[hidden_games] > user_mean

    neg_pred_games = hidden_games[np.argwhere(neg & (diff < ALLOW_ERR)).flatten()]
    pos_pred_games = hidden_games[np.argwhere(pos & (diff < ALLOW_ERR)).flatten()]

    # Find games ids
    neg_pred_games = games_table[games_table.index.isin(neg_pred_games)].values
    pos_pred_games =  games_table[games_table.index.isin(pos_pred_games)].values

    # Find users ids
    sim_users = users_table[users_table.index.isin(sim_users)].values
    sim_users_neg = comments_all[comments_all["Game id"].isin(neg_pred_games) & comments_all["User id"].isin(sim_users)]
    sim_users_pos = comments_all[comments_all["Game id"].isin(pos_pred_games) & comments_all["User id"].isin(sim_users)]
    print(pos_pred_games)
    user_pos = comments_all[(comments_all["Game id"].isin(neg_pred_games)) & (comments_all["User id"] == user_id)]
    user_neg = comments_all[(comments_all["Game id"].isin(pos_pred_games)) & (comments_all["User id"] == user_id)]

    print(f"User id : {user_id}, nb correct negative : {neg_pred_games.shape[0]}, nb correct positives : {pos_pred_games.shape[0]}")
    print(sim_users_neg.shape, sim_users_pos.shape, user_pos.shape, user_neg.shape)
    plot_barplots(sim_users_neg, sim_users_pos, user_neg, user_pos)


In [ ]:
# knn_comments new version
def plot_barplots(sim_users_neg, sim_users_pos, user_neg, user_pos, user_id): # user_id
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))
        # Negatives comments
        bigrams_neg = create_df(BigramCollocationFinder.from_documents(sim_users_neg["Lemma"].str.split().tolist()),
                                BigramAssocMeasures.raw_freq)

        bigrams_neg_user = create_df(BigramCollocationFinder.from_documents(user_neg["Lemma"].str.split().tolist()),
                                BigramAssocMeasures.raw_freq)
        
        bigrams_neg = bigrams_neg[bigrams_neg["Lemma"].isin(bigrams_neg_user["Lemma"])]

        # Find intersection
        bigrams_neg = bigrams_neg[bigrams_neg["Lemma"].isin(bigrams_neg_user["Lemma"])]
        bigrams_neg_user = bigrams_neg_user[bigrams_neg_user["Lemma"].isin(bigrams_neg["Lemma"])]

        # Positive comments
        bigrams_pos = create_df(BigramCollocationFinder.from_documents(sim_users_pos["Lemma"].str.split().tolist()),
                                BigramAssocMeasures.raw_freq)

        bigrams_pos_user = create_df( BigramCollocationFinder.from_documents(user_pos["Lemma"].str.split().tolist()),
                                BigramAssocMeasures.raw_freq)
        
        # Find intersection
        bigrams_pos = bigrams_pos[bigrams_pos["Lemma"].isin(bigrams_pos_user["Lemma"])]
        bigrams_pos_user = bigrams_pos_user[bigrams_pos_user["Lemma"].isin(bigrams_pos["Lemma"])]


        sns.barplot(data=bigrams_neg.sort_values(by="Freq", ascending=False).head(40), y="Lemma", x="Freq", ax=ax1)
        sns.barplot(data=bigrams_pos.sort_values(by="Freq", ascending=False).head(40), y="Lemma", x="Freq", ax=ax2)

        sns.barplot(data=bigrams_neg_user.sort_values(by="Freq", ascending=False).head(40), y="Lemma", x="Freq", ax=ax1, color="r", alpha=0.5)
        sns.barplot(data=bigrams_pos_user.sort_values(by="Freq", ascending=False).head(40), y="Lemma", x="Freq", ax=ax2, color="r", alpha=0.5)

        ax1.set_title(f"Negative bigrams for user {user_id} (id)")
        ax2.set_title(f"Positive bigrams for user {user_id} (id)")
        ax1.tick_params(axis='y', labelsize=8)
        ax2.tick_params(axis='y', labelsize=8)

        plt.tight_layout()
        return ax1, ax2

# type : simi, less_simi, random
def knn_comments(user_id, games_to_consider, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, type='simi', k=40):

    user_ind = users_table[users_table == user_id].index[0]
    games_to_hide = np.random.choice(games_to_consider, size=200, replace=False)

    hidden_games = np.intersect1d(games_table[games_table.isin(games_to_hide)].index, mask_ratings[user_ind, :].nonzero()[0])

    prev_ratings, prev_mask_ratings = matrix_ratings[user_ind, :], mask_ratings[user_ind, :], 
    prev_sim = cos_sim_matrix[user_ind, :]

    # hide games
    matrix_ratings[user_ind, hidden_games] = 0
    mask_ratings[user_ind, hidden_games] = 0

    recalc_cos_similarity(user_ind, matrix_ratings, cos_sim_matrix)

    # choice of similar users
    match type:
        case 'simi':
            sim_users =  get_KNN(cos_sim_matrix, k, user_ind)
        case 'less_simi':
            sim_users =  get_KNN(cos_sim_matrix, users_table.shape[0], user_ind)
            sim_users = sim_users[-k:]
        case 'random':
            sim_users =  get_KNN(cos_sim_matrix, users_table.shape[0], user_ind)
            sim_users = np.random.choice(sim_users, size=k, replace=False)
              
    print("similar users", sim_users, len(sim_users))
    pred_ratings, mask_pred_ratings = predict_ratings_baseline(matrix_ratings, mask_ratings,
                                                                sim_users, cos_sim_matrix, user_ind)
    
    # restore
    matrix_ratings[user_ind, :], mask_ratings[user_ind, :] = prev_ratings, prev_mask_ratings
    cos_sim_matrix[user_ind, :], cos_sim_matrix[:, user_ind] = prev_sim, prev_sim

    diff = np.abs(matrix_ratings[user_ind, hidden_games] - pred_ratings[hidden_games])

    ALLOW_ERR = 2
    user_mean = users_mean.loc[users_mean["User id"] == user_id, "Rating"].item()
    pos, neg = pred_ratings[hidden_games] < user_mean, pred_ratings[hidden_games] > user_mean

    neg_pred_games = hidden_games[np.argwhere(neg & (diff < ALLOW_ERR)).flatten()]
    pos_pred_games = hidden_games[np.argwhere(pos & (diff < ALLOW_ERR)).flatten()]

    # Find games ids
    neg_pred_games = games_table[games_table.index.isin(neg_pred_games)].values
    pos_pred_games =  games_table[games_table.index.isin(pos_pred_games)].values

    # Find users ids
    sim_users = users_table[users_table.index.isin(sim_users)].values
    sim_users_neg = comments_all[comments_all["Game id"].isin(neg_pred_games) & comments_all["User id"].isin(sim_users)]
    sim_users_pos = comments_all[comments_all["Game id"].isin(pos_pred_games) & comments_all["User id"].isin(sim_users)]
    print(pos_pred_games)
    user_pos = comments_all[(comments_all["Game id"].isin(neg_pred_games)) & (comments_all["User id"] == user_id)]
    user_neg = comments_all[(comments_all["Game id"].isin(pos_pred_games)) & (comments_all["User id"] == user_id)]

    print(f"User id : {user_id}, nb correct negative : {neg_pred_games.shape[0]}, nb correct positives : {pos_pred_games.shape[0]}")
    print(sim_users_neg.shape, sim_users_pos.shape, user_pos.shape, user_neg.shape)
    plot_barplots(sim_users_neg, sim_users_pos, user_neg, user_pos, user_id)


### User 208 (744), similar users 

In [ ]:
np.random.seed(90)
knn_comments(208, games_to_consider, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, type="simi")

### most distant users

In [ ]:
np.random.seed(90)
knn_comments(208, games_to_consider, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, type="less_simi")

### random users

In [ ]:
np.random.seed(90)
knn_comments(208, games_to_consider, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, type="random")

### User 1900 (220) similar users

In [ ]:
np.random.seed(90)
knn_comments(1900, games_to_consider, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, type="simi", k=40)

### most distant users

In [ ]:
np.random.seed(90)
knn_comments(1900, games_to_consider, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, type="less_simi", k=40)

### random users

In [ ]:
np.random.seed(90)
knn_comments(1900, games_to_consider, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, type="random", k=40)

### Using tf idf to filter bigrams

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
# lemmatized comments
all_doc = comments_all['Lemma']
vectorizer = TfidfVectorizer(ngram_range=(2, 2), min_df=5, max_df=0.8) # bigrams
vectors = vectorizer.fit_transform(all_doc)

In [ ]:
bigrams_ens = vectorizer.get_feature_names_out()

In [ ]:
comments_all = comments_all.drop(columns=['Comment line']).reset_index()
comments_all = comments_all.drop(columns=['index']).reset_index()
comments_all

In [ ]:
# using tf idf filtering
def plot_barplots(sim_users_neg, sim_users_pos, user_neg, user_pos, user_id, threshold): # user id 

        _, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))

        def f_all_comment(comment_grp): # filters the bigrams of comment using tf idf
            document = np.array([])

            for index, lem in zip(comment_grp['index'], comment_grp['Lemma']): 
                g = BigramCollocationFinder.from_words(lem.split()).score_ngrams(BigramAssocMeasures.raw_freq)
                values = vectors[index].data  # Non-zero values in the sparse matrix
                mask = values >= threshold
                values = values[mask]
                indices = vectors[index].indices[mask]
                keep_bigrams = bigrams_ens[indices[np.argsort(values)[::-1]]]
                kept = np.array([" ".join(bigram) for bigram, _ in g if " ".join(bigram) in keep_bigrams])
                if kept.size != 0:
                    document = np.concatenate((document,kept), axis = 0)
            return document        
        
        def filtered_big_df(df): # construct frequency lemma df
            bigrams_comments = df.groupby('User id').apply(f_all_comment,include_groups=False).reset_index(drop=True).values
            if bigrams_comments.size != 0:
                bigrams_comments = np.hstack(bigrams_comments)
            val, count = np.unique(bigrams_comments, return_counts=True)
            count = count/len(bigrams_comments)
            return pd.DataFrame({"Lemma": val, 'Freq': count})

        # Negatives comments
        bigrams_neg = filtered_big_df(sim_users_neg)        
        bigrams_neg_user = filtered_big_df(user_neg)

        # Find intersection
        bigrams_neg = bigrams_neg[bigrams_neg["Lemma"].isin(bigrams_neg_user["Lemma"])]
        bigrams_neg_user = bigrams_neg_user[bigrams_neg_user["Lemma"].isin(bigrams_neg["Lemma"])]

        # # Positive comments
        bigrams_pos = filtered_big_df(sim_users_pos)
        bigrams_pos_user = filtered_big_df(user_pos)
        
        # Find intersection
        bigrams_pos = bigrams_pos[bigrams_pos["Lemma"].isin(bigrams_pos_user["Lemma"])]
        bigrams_pos_user = bigrams_pos_user[bigrams_pos_user["Lemma"].isin(bigrams_pos["Lemma"])]

        sns.barplot(data=bigrams_neg.sort_values(by="Freq", ascending=False).head(40), y="Lemma", x="Freq", ax=ax1)
        sns.barplot(data=bigrams_pos.sort_values(by="Freq", ascending=False).head(40), y="Lemma", x="Freq", ax=ax2)

        sns.barplot(data=bigrams_neg_user.sort_values(by="Freq", ascending=False).head(40), y="Lemma", x="Freq", ax=ax1, color="r", alpha=0.5)
        sns.barplot(data=bigrams_pos_user.sort_values(by="Freq", ascending=False).head(40), y="Lemma", x="Freq", ax=ax2, color="r", alpha=0.5)

        ax1.set_title(f"Negative bigrams for user {user_id} (id)")
        ax2.set_title(f"Positive bigrams for user {user_id} (id)")
        ax1.tick_params(axis='y', labelsize=8)
        ax2.tick_params(axis='y', labelsize=8)

        plt.tight_layout()
        return ax1, ax2

# type: random, simi, less_simi
def knn_comments(user_id, games_to_consider, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, type = 'simi', threshold = 0, k = 40):    
    # rating matrix
    user_ind = users_table[users_table == user_id].index[0]
    games_to_hide = np.random.choice(games_to_consider, size=200, replace=False)
    hidden_games = np.intersect1d(games_table[games_table.isin(games_to_hide)].index, mask_ratings[user_ind, :].nonzero()[0])
    prev_ratings, prev_mask_ratings = matrix_ratings[user_ind, :], mask_ratings[user_ind, :], 
    prev_sim = cos_sim_matrix[user_ind, :]

    # hide games
    matrix_ratings[user_ind, hidden_games] = 0
    mask_ratings[user_ind, hidden_games] = 0

    recalc_cos_similarity(user_ind, matrix_ratings, cos_sim_matrix)

    # choice of neighbors
    match type:
        case 'simi':
            sim_users = get_KNN(cos_sim_matrix, k, user_ind)
        case 'less_simi':
            sim_users = get_KNN(cos_sim_matrix, users_table.shape[0], user_ind)[-k:] # furthest
        case 'random':
            sim_users = get_KNN(cos_sim_matrix, users_table.shape[0], user_ind) # random
            sim_users = np.random.choice(sim_users, size=k, replace=False)

    print('number of neighbors', len(sim_users))
            
    pred_ratings, mask_pred_ratings = predict_ratings_baseline(matrix_ratings, mask_ratings,
                                                                sim_users, cos_sim_matrix, user_ind)
    
    # restore
    matrix_ratings[user_ind, :], mask_ratings[user_ind, :] = prev_ratings, prev_mask_ratings
    cos_sim_matrix[user_ind, :], cos_sim_matrix[:, user_ind] = prev_sim, prev_sim

    diff = np.abs(matrix_ratings[user_ind, hidden_games] - pred_ratings[hidden_games])

    ALLOW_ERR = 2
    user_mean = users_mean.loc[users_mean["User id"] == user_id, "Rating"].item()
    pos, neg = pred_ratings[hidden_games] < user_mean, pred_ratings[hidden_games] > user_mean

    neg_pred_games = hidden_games[np.argwhere(neg & (diff < ALLOW_ERR)).flatten()]
    pos_pred_games = hidden_games[np.argwhere(pos & (diff < ALLOW_ERR)).flatten()]

    # Find games ids
    neg_pred_games = games_table[games_table.index.isin(neg_pred_games)].values
    pos_pred_games =  games_table[games_table.index.isin(pos_pred_games)].values

    # Find users ids
    sim_users = users_table[users_table.index.isin(sim_users)].values
    sim_users_neg = comments_all[comments_all["Game id"].isin(neg_pred_games) & comments_all["User id"].isin(sim_users)]
    sim_users_pos = comments_all[comments_all["Game id"].isin(pos_pred_games) & comments_all["User id"].isin(sim_users)]
    # print(pos_pred_games)
    user_pos = comments_all[(comments_all["Game id"].isin(neg_pred_games)) & (comments_all["User id"] == user_id)]
    user_neg = comments_all[(comments_all["Game id"].isin(pos_pred_games)) & (comments_all["User id"] == user_id)]

    # print(sim_users_neg)

    print(f"User id : {user_id}, nb correct negative : {neg_pred_games.shape[0]}, nb correct positives : {pos_pred_games.shape[0]}")
    print(sim_users_neg.shape, sim_users_pos.shape, user_pos.shape, user_neg.shape)
    plot_barplots(sim_users_neg, sim_users_pos, user_neg, user_pos, user_id, threshold)


### User 208 (744) similar users

In [ ]:
np.random.seed(90)
knn_comments(208, games_to_consider, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, type='simi', threshold=0.13, k = 40)

### most distant users

In [ ]:
np.random.seed(90)
knn_comments(208, games_to_consider, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, type='less_simi', threshold=0.13, k = 40)

### random users

In [ ]:
np.random.seed(90)
knn_comments(208, games_to_consider, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, type='random', threshold=0.13, k = 40)

### User 1900 (220) similar users

In [ ]:
np.random.seed(90)
knn_comments(83, games_to_consider, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, type='simi', threshold=0.13, k = 40)

### most distant users

In [ ]:
np.random.seed(90)
knn_comments(1900, games_to_consider, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, type='less_simi', threshold=0.13, k = 80)

### random users

In [ ]:
np.random.seed(90)
knn_comments(83, games_to_consider, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, type='random', threshold=0.13, k = 40)

### user 2431 (60) similar users

In [ ]:
np.random.seed(90)
knn_comments(2431, games_to_consider, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, type='simi', threshold=0.13, k = 40)

In [ ]:
np.random.seed(90)
knn_comments(2431, games_to_consider, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, type='less_simi', threshold=0.13, k = 40)

In [ ]:
np.random.seed(90)
knn_comments(2431, games_to_consider, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, type='random', threshold=0.13, k = 40)

---

In [ ]:
np.random.seed()
knn_similar_comments(91, games_to_consider, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table)

In [ ]:
fpos = words_freq(lemmas_pos, corpus)
fneg = words_freq(lemmas_neg, corpus)

fpos = fpos.sort_values(by=['Freq'], ascending = False)
fneg = fneg.sort_values(by=['Freq'], ascending = False)

fdiff = ft.diff_freq(fpos,fneg) # si valeur >= 0, alors + grande frequence dans fpos que fneg 
fdiff

In [ ]:
sns.set_theme(rc={'figure.figsize' : (12, 5)})
ax = sns.lineplot(fpos.head(90),x='Lemma',y='Freq',label='avis positifs')
ax = sns.lineplot(fneg.head(90),x='Lemma',y='Freq',label='avis négatifs')
ax.tick_params(axis='x', rotation=90, labelsize=8)
ax.set_title("Fréquence des Lemmas par type d'avis : corpus 5000")

On remarque que pour 2000 mots, tous les mots dans fpos sont aussi dans fneg

In [ ]:
sns.set_theme(rc={'figure.figsize' : (12, 5)})
ax = sns.lineplot(fdiff.head(90), x='Lemma',y='Freq differency', label='1')
ax.tick_params(axis='x', rotation=90, labelsize=8)
ax.set_title("Différence de fréquences : 50 mots + notables avis positifs : corpus 5000")

In [ ]:
sns.set_theme(rc={'figure.figsize' : (12, 5)})
ax = sns.lineplot(fdiff.tail(90)[::-1], x='Lemma',y='Freq differency', label='1')
ax.tick_params(axis='x', rotation=90, labelsize=8)
ax.set_title("Différence de fréquence : 50 mots + notables avis négatifs : corpus 5000")

Zoom sur les différences de fréquence


In [ ]:
lemmas_neg = fdiff.tail(90)['Lemma'].to_numpy()
pos_tail = fpos[fpos['Lemma'].isin(lemmas_neg)]
neg_tail = fneg[fneg['Lemma'].isin(lemmas_neg)]

ax = sns.lineplot(pos_tail, x='Lemma', y='Freq', label='avis positifs')
ax = sns.lineplot(neg_tail, x='Lemma', y='Freq', label='avis négatifs')
ax.tick_params(axis='x', rotation=90, labelsize=8)
ax.set_title("Différence de fréquence sur les pires mots : corpus 5000")

In [ ]:
lemmas_neg = fdiff.head(90)['Lemma'].to_numpy()
pos_head = fpos[fpos['Lemma'].isin(lemmas_neg)]
neg_head = fneg[fneg['Lemma'].isin(lemmas_neg)]

ax = sns.lineplot(pos_head, x='Lemma', y='Freq', label='avis positifs')
ax = sns.lineplot(neg_head, x='Lemma', y='Freq', label='avis négatifs')
ax.tick_params(axis='x', rotation=90, labelsize=8)
ax.set_title("Différence de fréquence sur les meilleurs mots : corpus 5000")

Visualisation par types de mots

In [ ]:
lemmas_sp = lemmas[~lemmas["Lemma"].isna()]
lemmas_sp = lemmas_sp[lemmas_sp['Part of speech'].isin(['ADJ', 'NOM', "VER", "NEG"])]
lemmas_sp_np = lemmas_sp[['Lemma', "Part of speech"]].to_numpy()

In [ ]:
lemma_speech = lemmas_sp[['Lemma', 'Part of speech']]
lemma_speech['Part of speech'].unique()

In [ ]:
# get only verbs
lemma_verb = lemma_speech[lemma_speech['Part of speech'].isin(['VER'])]
fdiff_verb = fdiff[fdiff['Lemma'].isin(lemma_verb['Lemma'].unique())].sort_values(by=['Freq differency'], ascending=False)
lemma_verb['Lemma'].unique()

In [ ]:
fdiff_verb.shape

In [ ]:
sns.set_theme(rc={'figure.figsize' : (12, 5)})
ax = sns.lineplot(fdiff_verb.head(90), x='Lemma',y='Freq differency', label='word frequence')
plt.title("Verbs most frequent in positive reviews")
ax.tick_params(axis='x', rotation=90, labelsize=8)

In [ ]:
sns.set_theme(rc={'figure.figsize' : (12, 5)})
ax = sns.lineplot(fdiff_verb.tail(90)[::-1], x='Lemma',y='Freq differency', label='word frequence')
plt.title("Verbs most frequent in negative reviews")
ax.tick_params(axis='x', rotation=90, labelsize=8)

In [ ]:
# get only adj
lemma_adj = lemma_speech[lemma_speech['Part of speech'].isin(['ADJ'])]
fdiff_adj = fdiff[fdiff['Lemma'].isin(lemma_adj['Lemma'].unique())].sort_values(by=['Freq differency'], ascending=False)
lemma_adj['Lemma'].unique()

In [ ]:
sns.set_theme(rc={'figure.figsize' : (12, 5)})
ax = sns.lineplot(fdiff_adj.head(90), x='Lemma',y='Freq differency', label='word frequence')
plt.title("Adj most frequent in positive reviews")
ax.tick_params(axis='x', rotation=90, labelsize=8)

In [ ]:
sns.set_theme(rc={'figure.figsize' : (12, 5)})
ax = sns.lineplot(fdiff_adj.tail(90)[::-1], x='Lemma',y='Freq differency', label='word frequence')
plt.title("Adj most frequent in negative reviews")
ax.tick_params(axis='x', rotation=90, labelsize=8)

# color one label
for label, position in zip(ax.get_xticklabels(), ax.get_xticks()):
    if label.get_text() == 'enfantin':
        label.set_color("blue")
        label.set_fontweight('bold')

In [ ]:
# get only nom
lemma_nom = lemma_speech[lemma_speech['Part of speech'].isin(['NOM'])]
fdiff_nom = fdiff[fdiff['Lemma'].isin(lemma_nom['Lemma'].unique())].sort_values(by=['Freq differency'], ascending=False)
lemma_nom['Lemma'].unique()

In [ ]:
sns.set_theme(rc={'figure.figsize' : (12, 5)})
ax = sns.lineplot(fdiff_nom.head(90), x='Lemma',y='Freq differency', label='word frequence')
plt.title("Nom most frequent in positive reviews")
ax.tick_params(axis='x', rotation=90, labelsize=8)

In [ ]:
sns.set_theme(rc={'figure.figsize' : (12, 5)})
ax = sns.lineplot(fdiff_nom.tail(90)[::-1], x='Lemma',y='Freq differency', label='word frequence')
plt.title("Nom most frequent in negative reviews")
ax.tick_params(axis='x', rotation=90, labelsize=8)